# PHATPSY 2.0: Interactive 3D Visualization of Hybridized Atomic Orbitals
**Valence-Bond / Perfect-Pairing Starting Orbitals & Volumetric 3D Isosurfaces**

*Jack A. Smith, PhD Dissertation (Univ. of Florida, 1980) modernized for PHATPSY 2.0*

---

## 1. Overview & Theoretical Context

In standard quantum chemistry codes and atomic basis libraries (Clementi & Roetti, Thakkar, ADF), basis functions and starting atomic orbitals are supplied as spherically symmetric or canonical isolated-atom ground states ($1s^2 2s^2 2p^2$ for Carbon, $1s^2 2s^2 2p^4$ for Oxygen). 

However, in molecular systems, atoms exist in **hybridized valence states**:
- **Oxygen in Water ($H_2O$)**: Possesses two lone pairs and two singly occupied bonding orbitals directed at the hydrogens with paired spin.
- **Carbon in Methane ($CH_4$)**: Promotes to an $sp^3$ valence state with four equivalent directed hybrid lobes pointing toward the vertices of a regular tetrahedron.
- **Carbon in Ethylene ($C_2H_4$)**: Forms three coplanar $sp^2$ hybrids ($120^\circ$) for $\sigma$-bonding, plus one pure unhybridized $2p_z$ orbital for $\pi$-bonding.
- **Carbon in Acetylene ($C_2H_2$)**: Forms two collinear $sp$ hybrids ($180^\circ$) for $\sigma$-bonding, plus two mutually orthogonal $\pi_x, \pi_y$ orbitals.

### The Modern `phatpsy.hybrids` Builder
In PHATPSY 2.0, `phatpsy.hybrids.build_atom_orbitals` constructs on-the-fly unitary transformations in the $(s, p_x, p_y, p_z)$ valence manifold:
$$\psi_i^{\text{hybrid}}(\mathbf{r}) = \sum_{\mu} C_{\mu i} \chi_\mu(\mathbf{r})$$
where $\chi_\mu(\mathbf{r})$ are real Slater-Type Orbitals (STOs):
$$\chi_{n,l,m}(\mathbf{r}) = N_{n,l,\zeta} r^{n-1} e^{-\zeta r} S_{l,m}(\theta, \phi)$$
with normalization $N_{n,l,\zeta} = \frac{(2\zeta)^{n + 1/2}}{\sqrt{(2n)!}}$ and real spherical harmonics $S_{l,m}$ in PHATPSY's engine convention ($m_l = 0 \to z, +1 \to x, -1 \to y$).

This notebook provides **interactive 3D volumetric renderings** of these hybridized atomic orbitals with:
- **Element Selector** (O, N, C, B, F, H, etc.)
- **Basis Set Selector** (DZ, ADF-DZ, ADF-TZP, ADF-TZ2P)
- **Geometry / Coordination Presets** (Water $C_{2v}$, Ammonia $C_{3v}$, Methane $T_d$, Ethylene $D_{2h}$, Acetylene $D_{\infty h}$, Diatomic $C_{\infty v}$)
- **Orbital Selector(s)** (Single hybrid orbital or composite multi-orbital renderings)
- **Isovalue Slider** (Wavefunction isosurface envelope threshold, e.g. $0.010 - 0.100$ a.u.)


### Pairing `AtomConfig.hybridization` with `AtomConfig.valence_config` (`ValenceOrbital`)
Alongside `AtomConfig.hybridization`, PHATPSY 2.0 provides **`AtomConfig.valence_config`** (`list[ValenceOrbital]`) to specify the per-orbital spin (`"up"`, `"down"`, `"none"`), occupation (`0`, `1`, `2`), chemical role (`"sigma"`, `"pi"`, `"delta"`, `"lone_pair"`, `"unhybridized"`), and bond target (`connect_to`) for every valence orbital on an atom:
- `AtomConfig.net_orbital_spin` evaluates $n_\uparrow - n_\downarrow$ across the atom's valence orbitals and reconciles `AtomConfig.spin` automatically (with zero-sum configurations defaulting to `"up"`).
- `CalculationDeck.physical_net_spin` sums `net_orbital_spin` over all atoms to enforce the exact molecular spin quantum number $2S$.
- `propose_valence_config(deck)` inspects molecular geometry, hybridization, and bipartite graph coloring to generate a complete `valence_config` dictionary automatically.

In [1]:
# Environment Setup & Core Imports
import os
import sys
import numpy as np
from IPython.display import display

# Ensure repository root is on PYTHONPATH
repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from phatpsy.basis import get_element_basis
from phatpsy.schema import ValenceOrbital
from phatpsy.builder import propose_valence_config
from phatpsy.hybrids import build_atom_orbitals, atom_valence_vectors, valence_space
from phatpsy.volumetric import compute_hybrid_orbital_grid, VolumetricGrid
from phatpsy.visualizer import (
    show_hybrid_orbitals,
    visualize_hybrid_orbitals,
    generate_hybrid_orbitals_visualization_data,
    classify_hybrid_orbitals,
    HYBRID_GEOMETRY_PRESETS,
)

print("PHATPSY 2.0 Hybridization & Volumetric Visualization Engine Loaded Successfully.")

PHATPSY 2.0 Hybridization & Volumetric Visualization Engine Loaded Successfully.


---
## 2. Interactive Orbital Explorer (Widget Interface)

Use the interactive controls below to dynamically explore hybridized atomic orbitals. 
- Change the **Element** and **Basis Set** to inspect how orbital shapes contract or expand.
- Select the **Geometry Preset** to switch between tetrahedral ($sp^3$), trigonal planar ($sp^2$), linear ($sp$), or bent coordination.
- Select **one or more Orbitals** to render single lobes or complete hybrid sets simultaneously.
- Adjust the **Isovalue Slider** to control the isosurface cutoff envelope (smaller = larger diffuse outer cloud; larger = dense inner core).


In [ ]:
# Interactive GUI with ipywidgets (with automatic fallback to preset rendering)
import os
import sys
repo_root = os.path.abspath(".." )
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from phatpsy.visualizer import (
    show_hybrid_orbitals,
    visualize_hybrid_orbitals,
    classify_hybrid_orbitals,
    HYBRID_GEOMETRY_PRESETS,
)
from phatpsy.basis import get_element_basis
from phatpsy.hybrids import build_atom_orbitals

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

if HAS_WIDGETS:
    # 1. UI Control Widgets
    element_dropdown = widgets.Dropdown(
        options=["O", "N", "C", "B", "F", "P", "S", "Ni", "Pt", "Fe", "Cu"],
        value="O",
        description="Element:",
        style={"description_width": "110px"},
        layout=widgets.Layout(width="260px")
    )
    
    basis_dropdown = widgets.Dropdown(
        options=["DZP", "DZ", "ADF-DZ", "ADF-TZP", "ADF-TZ2P"],
        value="DZP",
        description="Basis Set:",
        style={"description_width": "110px"},
        layout=widgets.Layout(width="260px")
    )
    
    geometry_dropdown = widgets.Dropdown(
        options=[
            ("--- Named Hybrid Types ---", "sp3"),
            ("sp³ Tetrahedral (Td)", "sp3"),
            ("sp² Trigonal Planar (D3h)", "sp2"),
            ("sp Linear (Dinfh)", "sp"),
            ("sp³ Trigonal Pyramidal (C3v)", "sp3_pyramidal"),
            ("sp³ Bent (C2v)", "sp3_bent"),
            ("sp² Bent (C2v)", "sp2_bent"),
            ("Pure p Orthogonal (C3v)", "pure_p"),
            ("dsp² Square Planar (D4h, Ni/Pt)", "dsp2"),
            ("d²sp³ Octahedral (Oh, Fe/Ni)", "d2sp3"),
            ("δ Delta Bonding (Ni/Cr/Mo)", "delta"),
            ("--- Molecular Presets ---", "water"),
            ("Water (H2O, C2v)", "water"),
            ("Ammonia (NH3, C3v)", "ammonia"),
            ("Methane (CH4, Td)", "methane"),
            ("Ethylene (C2H4, D2h)", "ethylene"),
            ("Acetylene (C2H2, Dinfh)", "acetylene"),
            ("Carbon Monoxide (CO, Cinfv, sp Neutral Dative-Pi)", "carbon_monoxide"),
            ("Diatomic (Cinfv)", "diatomic"),
        ],
        value="water",
        description="Geometry:",
        style={"description_width": "110px"},
        layout=widgets.Layout(width="300px")
    )
    
    isovalue_slider = widgets.FloatSlider(
        value=0.10,
        min=0.010,
        max=0.100,
        step=0.005,
        description="Isovalue (au):",
        readout_format=".3f",
        style={"description_width": "110px"},
        layout=widgets.Layout(width="380px")
    )
    
    both_phases_checkbox = widgets.Checkbox(
        value=True,
        description="Show Dual Phase (+/-)",
        style={"description_width": "0px"},
        layout=widgets.Layout(width="200px")
    )
    
    orbital_selector = widgets.SelectMultiple(
        description="Orbitals:",
        options=[],
        style={"description_width": "110px"},
        layout=widgets.Layout(width="480px", height="130px")
    )
    
    render_button = widgets.Button(
        description="Render Inline",
        button_style="primary",
        icon="cube",
        layout=widgets.Layout(width="170px", height="38px")
    )

    browser_button = widgets.Button(
        description="Open in Browser ↗",
        button_style="info",
        icon="external-link",
        layout=widgets.Layout(width="180px", height="38px")
    )
    
    info_output = widgets.HTML(value="")
    viewport_output = widgets.Output()

    def update_orbital_options(*args):
        elem_sym = element_dropdown.value
        geom_key = geometry_dropdown.value
        basis_name = basis_dropdown.value
        preset = HYBRID_GEOMETRY_PRESETS.get(geom_key, HYBRID_GEOMETRY_PRESETS['water'])
        if preset.get('default_element') == 'Ni' and elem_sym not in ('Ni', 'Pt', 'Fe', 'Cu'):
            element_dropdown.value = 'Ni'
            elem_sym = 'Ni'
        try:
            elem_data = get_element_basis(elem_sym, basis_name)
            delta_dirs = preset.get("delta_dirs", ())
            hyb_name = preset.get("hybridization")
            orbs = build_atom_orbitals(
                elem=elem_data,
                spin="up",
                sigma_dirs=preset["sigma_dirs"],
                pi_dirs=preset["pi_dirs"],
                hybridization=hyb_name,
                delta_dirs=delta_dirs,
            )
            partner_specs = preset.get("partners") or [{"element": preset.get("partner_element", "H")}] * len(preset["sigma_dirs"])
            partner_labels = [f"{elem_sym}-{p.get('element', 'H')}" for p in partner_specs]
            labels = classify_hybrid_orbitals(
                orbs,
                len(preset["sigma_dirs"]),
                len(preset["pi_dirs"]),
                partner_names=partner_labels,
                n_delta=len(delta_dirs),
            )
            orbital_selector.options = [(f"{idx}: {lbl}", idx) for idx, lbl in enumerate(labels)]
            orbital_selector.value = (1,) if len(orbs) > 1 else (0,)
            info_output.value = f"<span style='color:#38bdf8; font-weight:600;'>{elem_sym}</span> in <span style='color:#a855f7; font-weight:600;'>{preset['name']}</span>: {len(orbs)} orbitals generated over {len(elem_data.functions)} STO basis functions."
        except Exception as err:
            info_output.value = f"<span style='color:#ef4444;'>Error: {err}</span>"

    def on_render_clicked(b=None):
        with viewport_output:
            clear_output(wait=True)
            sel = list(orbital_selector.value) or [1]
            try:
                display(show_hybrid_orbitals(
                    element=element_dropdown.value,
                    basis=basis_dropdown.value,
                    geometry=geometry_dropdown.value,
                    selected_orbitals=sel,
                    isovalue=isovalue_slider.value,
                    show_both_phases=both_phases_checkbox.value,
                    height=560,
                ))
            except Exception as e:
                print(f"Rendering failed: {e}")

    def on_browser_clicked(b=None):
        sel = list(orbital_selector.value) or [1]
        try:
            visualize_hybrid_orbitals(
                element=element_dropdown.value,
                basis=basis_dropdown.value,
                geometry=geometry_dropdown.value,
                selected_orbitals=sel,
                isovalue=isovalue_slider.value,
                show_both_phases=both_phases_checkbox.value,
                auto_open=True,
            )
            elem_sym = element_dropdown.value.strip().capitalize()
            info_output.value = (
                f"<span style='color:#38bdf8; font-weight:600;'>🌐 Launched full-screen 3D WebGL viewer for "
                f"<b>{elem_sym}</b> in default web browser.</span>"
            )
        except Exception as e:
            info_output.value = f"<span style='color:#ef4444;'>Browser launch failed: {e}</span>"

    element_dropdown.observe(update_orbital_options, names="value")
    geometry_dropdown.observe(update_orbital_options, names="value")
    basis_dropdown.observe(update_orbital_options, names="value")
    render_button.on_click(on_render_clicked)
    browser_button.on_click(on_browser_clicked)

    update_orbital_options()
    
    controls_ui = widgets.VBox([
        widgets.HBox([element_dropdown, basis_dropdown]),
        widgets.HBox([geometry_dropdown, both_phases_checkbox]),
        orbital_selector,
        isovalue_slider,
        widgets.HBox([render_button, browser_button], layout=widgets.Layout(margin="8px 0 8px 110px", gap="12px")),
        info_output,
        viewport_output,
    ])
    display(controls_ui)
    on_render_clicked()

else:
    print("ipywidgets is not currently installed in the kernel environment.")
    print("Displaying standalone interactive 3D WebGL rendering of Oxygen in Water (H2O, DZP basis):")
    display(show_hybrid_orbitals("O", geometry="water", selected_orbitals=[3], isovalue=0.040))

---
## 3. Case Studies: Prototypical Hybrid Geometries

Let's examine how the directed-hybrid builder operates on standard molecular archetypes.

### 3.1 Water ($H_2O$, $C_{2v}$): $\sigma$-Bonds vs. Lone Pairs

In water, Oxygen has 8 electrons:
- $1s^2$ core ($[1.0, 1.0]$, $\epsilon \approx -20.66$ Ha)
- Two non-bonding Lone Pairs ($[1.0, 1.0]$, $\epsilon \approx -0.63$ and $-0.95$ Ha)
- Two bonding $\sigma(O-H)$ hybrids ($[1.0, 0.0]$, $\epsilon = -0.772$ Ha) pointing directly toward the hydrogens along the $104.5^\circ$ bond angle.

Below, we visualize **both $\sigma(O-H)$ bonding hybrids together** to verify their spatial orientation along the $O-H$ bond axes:


In [6]:
from phatpsy.visualizer import show_hybrid_orbitals, visualize_hybrid_orbitals
# Oxygen in Water: Display both sigma-bonding hybrids simultaneously
visualize_hybrid_orbitals(
    element="O",
    geometry="water",
    selected_orbitals=[3, 4],  # sigma_1 and sigma_2
    isovalue=0.10,
    show_both_phases=True,
    title="Oxygen in Water: Both O-H Sigma Bonding Hybrids (104.5 deg)"
)


'<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <title>Oxygen in Water: Both O-H Sigma Bonding Hybrids (104.5 deg) - PHATPSY 3D Viewer &amp; Editor</title>\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  <style>\n    * { box-sizing: border-box; margin: 0; padding: 0; }\n    body {\n      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;\n      background: #0f172a;\n      color: #f8fafc;\n      overflow: hidden;\n      width: 100vw;\n      height: 100vh;\n    }\n    #canvas-container {\n      width: 100%;\n      height: 100%;\n      position: absolute;\n      top: 0;\n      left: 0;\n    }\n    #top-bar {\n      position: absolute;\n      top: 0;\n      left: 0;\n      width: 100vw;\n      height: 48px;\n      background: rgba(15, 23, 42, 0.88);\n      backdrop-filter: blur(16px);\n      -webkit-backdrop-filter: blur(16px);\n      border-bottom: 1px solid rgba(255, 255, 255, 0.12);\n    

Now let's examine Oxygen's **two non-bonding lone pairs** in water:
- **Lone Pair 1 (Orbital 1)**: Pure out-of-plane $p_y$ character ($\pi$-like non-bonding lone pair).
- **Lone Pair 2 (Orbital 2)**: In-plane $s-p_z$ hybrid pointing opposite the $H-O-H$ vertex.


In [9]:
from phatpsy.visualizer import show_hybrid_orbitals, visualize_hybrid_orbitals
# Oxygen in Water: Display both Lone Pairs simultaneously
visualize_hybrid_orbitals(
    element="O",
    geometry="water",
    selected_orbitals=[2],  # Lone Pair 1 and Lone Pair 2
    isovalue=0.040,
    show_both_phases=True,
    title="Oxygen in Water: Non-Bonding Lone Pairs"
)


'<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <title>Oxygen in Water: Non-Bonding Lone Pairs - PHATPSY 3D Viewer &amp; Editor</title>\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  <style>\n    * { box-sizing: border-box; margin: 0; padding: 0; }\n    body {\n      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;\n      background: #0f172a;\n      color: #f8fafc;\n      overflow: hidden;\n      width: 100vw;\n      height: 100vh;\n    }\n    #canvas-container {\n      width: 100%;\n      height: 100%;\n      position: absolute;\n      top: 0;\n      left: 0;\n    }\n    #top-bar {\n      position: absolute;\n      top: 0;\n      left: 0;\n      width: 100vw;\n      height: 48px;\n      background: rgba(15, 23, 42, 0.88);\n      backdrop-filter: blur(16px);\n      -webkit-backdrop-filter: blur(16px);\n      border-bottom: 1px solid rgba(255, 255, 255, 0.12);\n      display: flex;\n  

### 3.2 Methane ($CH_4$, $T_d$): Canonical Tetrahedral $sp^3$ Hybrids

In methane, Carbon has 4 valence electrons that hybridize into 4 equivalent $sp^3$ orbitals pointing at the vertices of a tetrahedron ($	heta = 109.47^\circ$).
All 4 hybrids are strictly degenerate with $\epsilon = -0.5006$ Ha.

By setting `show_both_phases=False`, we display the **positive lobes of all 4 tetrahedral hybrids at once**, showing the perfect tetrahedral geometry:


In [10]:
from phatpsy.visualizer import show_hybrid_orbitals, visualize_hybrid_orbitals
# Carbon in Methane: All 4 Tetrahedral sp3 Hybrids
visualize_hybrid_orbitals(
    element="C",
    geometry="methane",
    selected_orbitals=[1],  # All 4 sp3 hybrids
    show_both_phases=False,         # Positive lobes to clearly see the tetrahedral axes
    title="Carbon in Methane: 4 Equivalent Tetrahedral sp3 Hybrids (Td)"
)


'<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <title>Carbon in Methane: 4 Equivalent Tetrahedral sp3 Hybrids (Td) - PHATPSY 3D Viewer &amp; Editor</title>\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  <style>\n    * { box-sizing: border-box; margin: 0; padding: 0; }\n    body {\n      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;\n      background: #0f172a;\n      color: #f8fafc;\n      overflow: hidden;\n      width: 100vw;\n      height: 100vh;\n    }\n    #canvas-container {\n      width: 100%;\n      height: 100%;\n      position: absolute;\n      top: 0;\n      left: 0;\n    }\n    #top-bar {\n      position: absolute;\n      top: 0;\n      left: 0;\n      width: 100vw;\n      height: 48px;\n      background: rgba(15, 23, 42, 0.88);\n      backdrop-filter: blur(16px);\n      -webkit-backdrop-filter: blur(16px);\n      border-bottom: 1px solid rgba(255, 255, 255, 0.12);\n   

### 3.3 Ethylene ($C_2H_4$, $D_{2h}$): Trigonal Planar $sp^2$ Hybrids + Out-of-Plane $\pi$ Orbital

In ethylene, each Carbon forms:
- Three coplanar $sp^2$ hybrids ($120^\circ$) for $\sigma$-bonding in the molecular plane.
- One unhybridized pure $2p_z$ orbital oriented perpendicular to the plane for $\pi$-bonding.

Below, we visualize one in-plane $sp^2$ hybrid alongside the out-of-plane $\pi$ orbital:


In [12]:
from phatpsy.visualizer import show_hybrid_orbitals, visualize_hybrid_orbitals
# Carbon in Ethylene: in-plane sp2 hybrid + out-of-plane pi lobe
visualize_hybrid_orbitals(
    element="C",
    geometry="ethylene",
    selected_orbitals=[2],  # pi orbital and in-plane sigma hybrid
    isovalue=0.035,
    show_both_phases=True,
    title="Carbon in Ethylene: sp2 Sigma Hybrid and Out-of-Plane Pi Orbital"
)


'<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <title>Carbon in Ethylene: sp2 Sigma Hybrid and Out-of-Plane Pi Orbital - PHATPSY 3D Viewer &amp; Editor</title>\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  <style>\n    * { box-sizing: border-box; margin: 0; padding: 0; }\n    body {\n      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;\n      background: #0f172a;\n      color: #f8fafc;\n      overflow: hidden;\n      width: 100vw;\n      height: 100vh;\n    }\n    #canvas-container {\n      width: 100%;\n      height: 100%;\n      position: absolute;\n      top: 0;\n      left: 0;\n    }\n    #top-bar {\n      position: absolute;\n      top: 0;\n      left: 0;\n      width: 100vw;\n      height: 48px;\n      background: rgba(15, 23, 42, 0.88);\n      backdrop-filter: blur(16px);\n      -webkit-backdrop-filter: blur(16px);\n      border-bottom: 1px solid rgba(255, 255, 255, 0.12);\

### 3.4 Acetylene ($C_2H_2$, $D_{\infty h}$): Linear $sp$ Hybrids + Two Orthogonal $\pi$ Bonds

In acetylene, Carbon forms:
- Two linear $sp$ hybrids directed along $+z$ and $-z$ ($180^\circ$).
- Two mutually orthogonal $\pi_x$ and $\pi_y$ orbitals forming the cylindrical electron barrel of the triple bond.


In [13]:
from phatpsy.visualizer import show_hybrid_orbitals, visualize_hybrid_orbitals
# Carbon in Acetylene: Both Orthogonal Pi Orbitals (pi_x and pi_y)
visualize_hybrid_orbitals(
    element="C",
    geometry="acetylene",
    selected_orbitals=[3, 4],  # pi_x and pi_y
    isovalue=0.035,
    show_both_phases=True,
    title="Carbon in Acetylene: Orthogonal Pi Orbitals (pi_x and pi_y)"
)


'<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <title>Carbon in Acetylene: Orthogonal Pi Orbitals (pi_x and pi_y) - PHATPSY 3D Viewer &amp; Editor</title>\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  <style>\n    * { box-sizing: border-box; margin: 0; padding: 0; }\n    body {\n      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;\n      background: #0f172a;\n      color: #f8fafc;\n      overflow: hidden;\n      width: 100vw;\n      height: 100vh;\n    }\n    #canvas-container {\n      width: 100%;\n      height: 100%;\n      position: absolute;\n      top: 0;\n      left: 0;\n    }\n    #top-bar {\n      position: absolute;\n      top: 0;\n      left: 0;\n      width: 100vw;\n      height: 48px;\n      background: rgba(15, 23, 42, 0.88);\n      backdrop-filter: blur(16px);\n      -webkit-backdrop-filter: blur(16px);\n      border-bottom: 1px solid rgba(255, 255, 255, 0.12);\n    

---
## 4. Basis Set Comparison: Clementi DZ vs. ADF-TZ2P

Because a hybrid orbital is a linear combination over the atom's own valence Slater-type basis functions, `phatpsy.hybrids` works seamlessly across all basis sets. 

Comparing Clementi double-zeta (`DZ`) against triple-zeta polarized (`ADF-TZ2P`):
- `DZ`: 5 STO functions per atom, minimal radial flexibility.
- `ADF-TZ2P`: 14 STO functions per atom (including diffuse functions and polarization $d$-orbitals), yielding smoother, more radially extensive lobes.


In [17]:
from IPython.display import display
from phatpsy.visualizer import show_hybrid_orbitals, visualize_hybrid_orbitals
# Compare Nitrogen in Ammonia: Clementi DZ vs ADF-TZ2P
print("--- Clementi DZ Ammonia Hybrid ---")
display(show_hybrid_orbitals("N", basis="DZ", geometry="ammonia", selected_orbitals=[2], isovalue=0.10, height=600, title="Ammonia N-H Sigma Hybrid (Clementi DZ)"))

print("--- ADF-TZ2P Ammonia Hybrid ---")
display(show_hybrid_orbitals("N", basis="ADF-TZ2P", geometry="ammonia", selected_orbitals=[2], isovalue=0.10, height=600, title="Ammonia N-H Sigma Hybrid (ADF-TZ2P Triple-Zeta Polarized)"))


--- Clementi DZ Ammonia Hybrid ---


--- ADF-TZ2P Ammonia Hybrid ---


---
## 5. Programmatic Custom Geometry & Mixed Bonding Partners

You can also pass arbitrary custom $\vec{\sigma}$ and $\vec{\pi}$ direction vectors, along with an explicit list of **mixed bonding partners** (where each bond can have a different element and bond length), to model complex coordination environments, asymmetric functional groups, or surface adsorption sites.

For example, consider a Carbon atom bonded to Nitrogen (-N$ single bond, 1.45 Å), Hydrogen (-H$, 1.09 Å), and Fluorine (-F$, 1.35 Å):


In [20]:
import numpy as np
from phatpsy.visualizer import show_hybrid_orbitals, visualize_hybrid_orbitals

# Custom arbitrary bonding directions (trigonal arrangement)
custom_sigma = [
    np.array([1.0, 0.0, 0.0]),          # Bond 1 along +x
    np.array([-0.5, np.sqrt(3)/2, 0.0]), # Bond 2 at 120 deg
    np.array([-0.5, -np.sqrt(3)/2, 0.0]),# Bond 3 at 240 deg
]
custom_pi = [
    np.array([0.0, 0.0, 1.0]),          # Out-of-plane pi lobe along z
]

# Explicit mixed partners: each bond has its own element and bond length
custom_partners = [
    ("N", 1.45),  # Partner 1: Nitrogen at 1.45 Angstrom
    ("H", 1.09),  # Partner 2: Hydrogen at 1.09 Angstrom
    ("F", 1.35),  # Partner 3: Fluorine at 1.35 Angstrom
]

show_hybrid_orbitals(
    element="C",
    basis="ADF-TZP",
    geometry=(custom_sigma, custom_pi),
    partners=custom_partners,
    selected_orbitals=[2, 3],  # sigma(C-N) and sigma(C-H)
    isovalue=0.040,
    height=480,
    title="Carbon: Mixed Partners (N, H, F) with Asymmetric Bond Lengths"
)


'<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <title>Oxygen: Custom 90-degree Corner Bonding Environment - PHATPSY 3D Viewer &amp; Editor</title>\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  <style>\n    * { box-sizing: border-box; margin: 0; padding: 0; }\n    body {\n      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;\n      background: #0f172a;\n      color: #f8fafc;\n      overflow: hidden;\n      width: 100vw;\n      height: 100vh;\n    }\n    #canvas-container {\n      width: 100%;\n      height: 100%;\n      position: absolute;\n      top: 0;\n      left: 0;\n    }\n    #top-bar {\n      position: absolute;\n      top: 0;\n      left: 0;\n      width: 100vw;\n      height: 48px;\n      background: rgba(15, 23, 42, 0.88);\n      backdrop-filter: blur(16px);\n      -webkit-backdrop-filter: blur(16px);\n      border-bottom: 1px solid rgba(255, 255, 255, 0.12);\n      displa

---
## 6. Transition Metal Hybridization & Delta Bonding ($dsp^2, d^2sp^3, \delta$)

In transition metals and surface science (a central mission of PHATPSY 2.0), valence $d$-orbitals actively participate in chemical bonding:
- **Square Planar ($dsp^2$, $D_{4h}$)**: 4 in-plane $\sigma$-bonds at $90^\circ$ (e.g. $[NiCl_4]^{2-}$, $Pt(II)$ complexes, 4-fold hollow surface sites) leaving axial $d_{z^2}$ and $d_{xy}, \pi$ non-bonding lone pairs.
- **Octahedral ($d^2sp^3$, $O_h$)**: 6 orthogonal $\sigma$-bonds along $\pm x, \pm y, \pm z$ leaving non-bonding $t_{2g}$ orbitals ($d_{xy}, d_{xz}, d_{yz}$).
- **$\delta$-Bonding**: Face-to-face 4-lobe overlap between $d_{x^2-y^2}$ and $d_{xy}$ orbitals possessing **two nodal planes** containing the internuclear axis (observed in metal-metal quadruple/quintuple bonds and surface bridge sites).

Below, we programmatically construct and visualize a square planar $dsp^2$ Nickel complex with 4 Chlorine ligands, as well as a $\delta$-bonding orbital:


In [ ]:
from IPython.display import display
from phatpsy.visualizer import show_hybrid_orbitals, visualize_hybrid_orbitals

# 1. Square Planar Ni(II) with 4 Chlorine ligands (dsp2)
print("Rendering Square Planar Ni (dsp2) with 4 Chlorine partners...")
display(show_hybrid_orbitals(
    element="Ni",
    basis="ADF-DZ",
    geometry="dsp2",
    partners=[("Cl", 2.20), ("Cl", 2.20), ("Cl", 2.20), ("Cl", 2.20)],
    selected_orbitals=[5, 6, 7, 8],
    isovalue=0.035,
    title="Nickel (Ni) dsp² Square Planar Hybrids with 4 Cl Partners",
    height=540,
))


In [ ]:
# 2. Delta Bonding Orbital (dx2-y2 / dxy with 2 nodal planes)
print("Rendering Delta-bonding orbital on Ni...")
display(show_hybrid_orbitals(
    element="Ni",
    basis="ADF-DZ",
    geometry="delta",
    selected_orbitals=[13],
    isovalue=0.030,
    title="Nickel (Ni) Delta-Bonding Orbital (Two Nodal Planes)",
    height=500,
))


---
## 7. Summary & Key Takeaways

1. **Predefined Named Hybridization Registry**: Users can assign standard hybridization types by name (`sp3`, `sp2`, `sp`, `sp3_pyramidal`, `sp3_bent`, `dsp2`, `d2sp3`, `delta`) to atoms in python decks or YAML files.
2. **Automatic Best-Fit Frame Alignment**: The Kabsch algorithm rotates the canonical hybrid frame to maximize directional overlap with actual bonded neighbor positions, eliminating manual coordinate transformations.
3. **Transition Metal & d-Orbital Generalization**: Expands the valence manifold to 9D ($s, p, d$), natively enabling square-planar ($dsp^2$), octahedral ($d^2sp^3$), and $\delta$-bonding starting orbitals.
4. **Particle Conservation & Invariants**: Gross populations and electron counts strictly match the element valence and oxidation state, preserving total molecular electron conservation.
5. **Interactive 3D Dual-Phase Visualization**: Volumetric marching-cubes rendering accurately depicts bonding and nodal structure, providing immediate visual verification of starting orbital orientation.
6. **Orbital-Resolved Valence Configurations (`AtomConfig.valence_config` & `propose_valence_config`)**: Encodes per-orbital spin (`up`/`down`/`none`), role (`orbital_type`: `sigma`/`pi`/`delta`/`lone_pair`/`unhybridized`), `occupation` (`0`/`1`/`2`), and bond connectivity (`connect_to`) with automatic eleven-point physical validation.